# Assignment 1: who will say yes to each other?

**Matias Arevalo**

**Business question.** A speed-dating organiser receives everyone's sign-up survey before the night. When the night is too short for everyone to meet everyone, which pairs should meet? I predict, *before the event*, the probability that a woman and a man both say yes to each other (a mutual match), and use that probability to decide who gets seated together.

**How the notebook is built.** Every step only adds to the one before it; nothing is cleaned in one place and undone in another.

| Section | What happens |
|---|---|
| 1 | Setup |
| 2 | Raw data: what is in it and which decisions it forces |
| 3 | One row per woman-man pair, with the mutual-match target |
| 4 | Features that are known before the night |
| 5 | Split by event: past events train, later events test |
| 6 | Regularisation strength and a check of the feature groups |
| 7 | The same logistic regression three ways: scikit-learn, a manual PyTorch loop, `nn.Module` + `torch.optim` |
| 8 | Test events: metrics, calibration and business cost in euros |
| 9 | Extra analyses: seating planner, what attraction data would add, how much data is enough |
| 10 | Save the models and results |
| 11 | Gradio dashboard, built only from the saved files |

Two rules hold throughout:
1. **No leakage through cleaning.** Missing answers stay missing until the model pipeline fills them with medians learned on the training events only. Nothing is imputed or scaled on the full data.
2. **Only pre-event information enters the model.** Ratings given during the date are kept aside for one "what if" analysis and never used as inputs.

**To run on Colab:** upload `Speed Dating Data.csv` to `/content` (Files panel on the left), then *Runtime -> Run all* (about 3 minutes).

## 1. Setup

In [ ]:
# Colab already has numpy, pandas, scikit-learn, scipy, torch, matplotlib and plotly; only Gradio is missing.
%pip install -q "gradio>=6,<7"

In [ ]:
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linprog
from scipy.sparse import coo_matrix
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import KFold, LeaveOneGroupOut, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CANDIDATES = [Path("/content/Speed Dating Data.csv"), Path("Speed Dating Data.csv"), Path("data/Speed Dating Data.csv")]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Upload 'Speed Dating Data.csv' to /content (Files panel) and run again.")

ART = Path("artifacts")   # models and results the dashboard loads
FIG = Path("figures")     # static copies of the figures
ART.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
C3 = {"sklearn": "#2a6fdb", "manual": "#e8833a", "module": "#2ca58d"}
LABEL = {"sklearn": "scikit-learn", "manual": "PyTorch manual loop", "module": "PyTorch nn.Module + Adam"}

summary: dict = {}   # every number the dashboard shows is collected here and saved in section 10
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:6.1f}s] {msg}", flush=True)

## 2. The raw data

Each row is **one person's record of one 4-minute date**: who they are (`iid`), who they met (`pid`), the event (`wave`), their sign-up survey, the scorecard they filled in during the date and their decision (`dec`). `match` is 1 when both said yes.

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding="unicode_escape")
people = raw.drop_duplicates("iid")    # sign-up answers are repeated on every row of a person; one row each

print(f"{len(raw):,} date records, {raw['iid'].nunique()} people "
      f"({(people['gender'] == 0).sum()} women, {(people['gender'] == 1).sum()} men), {raw['wave'].nunique()} events")
print(f"records without a partner id (cannot be paired): {raw['pid'].isna().sum()}")
keys = set(zip(raw["iid"], raw["pid"]))
print(f"records whose mirror record (partner's view of the same date) exists: "
      f"{np.mean([(p, i) in keys for i, p in zip(raw['iid'], raw['pid'])]):.1%}")
raw[["iid", "pid", "wave", "gender", "age", "attr1_1", "attr3_1", "attr", "like", "dec", "match"]].head()

Every date is in the file twice, once from her side and once from his. The target needs both decisions, so section 3 joins the two records into one row per pair.

### 2.1 Which columns may the model use?

The test I apply: *could the organiser have this number the day before the event?*

| Column family | Example | When it is collected | Used as input? |
|---|---|---|---|
| Sign-up survey (`*_1`, demographics) | `age`, `goal`, interests, `attr1_1` (what I look for), `attr3_1` (how I rate myself) | before the event | **yes** |
| Event plan | `round` (people met that night), `samerace` | before the event | **yes** |
| Scorecard | `attr`, `fun`, `like`, `dec` | during the date | no (only in the "what if" analysis, 9b) |
| Follow-up surveys (`*_2`, `*_3`, `*_s`) | `attr1_2` | after the event | no |

### 2.2 Missing sign-up answers
Measured per person (each person counts once, however many dates they had).

In [ ]:
INTERESTS_ALL = ["sports", "exercise", "museums", "art", "hiking", "gaming", "reading",
                 "tv", "theater", "movies", "concerts", "music", "shopping"]
ATTRS6 = ["attr", "sinc", "intel", "fun", "amb", "shar"]   # attractive, sincere, intelligent, fun, ambitious, shared interests
ATTRS5 = ATTRS6[:5]                                         # self-ratings have no "shared interests" item

signup_blocks = {
    "profile (age, field, goal, dating, going out, happiness, race/religion importance)":
        ["age", "field_cd", "goal", "date", "go_out", "exphappy", "imprace", "imprelig"],
    "interests (13 x 1-10)": INTERESTS_ALL,
    "1_1 what I look for (100 points)": [f"{a}1_1" for a in ATTRS6],
    "2_1 what I think the other side looks for": [f"{a}2_1" for a in ATTRS6],
    "3_1 how I rate myself (1-10)": [f"{a}3_1" for a in ATTRS5],
    "4_1 what I think my own sex looks for": [f"{a}4_1" for a in ATTRS6],
    "5_1 how I think others rate me": [f"{a}5_1" for a in ATTRS5],
}
miss = pd.Series({name: people[cols].isna().any(axis=1).mean() * 100 for name, cols in signup_blocks.items()})
miss_by_wave = pd.DataFrame({b: people.groupby("wave")[signup_blocks[b]].apply(lambda d: d.isna().any(axis=1).mean())
                             for b in list(signup_blocks)[-2:]})

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1.1, 1]})
miss.iloc[::-1].plot.barh(ax=ax[0], color=["#c44" if v > 10 else "#2a6fdb" for v in miss.iloc[::-1]])
ax[0].set(xlabel="% of people with at least one missing answer", title="Missing sign-up answers, by question block")
(miss_by_wave * 100).plot(ax=ax[1], marker="o")
ax[1].set(xlabel="wave", ylabel="% of people missing", title="4_1 and 5_1: missing for whole waves")
ax[1].legend([b.split(" ")[0] for b in miss_by_wave.columns], frameon=False)
fig.tight_layout()
fig.savefig(FIG / "eda_missing.png")
plt.show()

**Decision.** Blocks `4_1` and `5_1` are missing for whole events (`4_1` was not asked in waves 1-5, `5_1` not in waves 1-9), so they are not used: the training events would have to be filled entirely by imputation. Every other block is missing for under 2% of people; those gaps stay `NaN` here and are filled later with medians from the training events only.

### 2.3 Interests that measure the same thing
Thirteen interest ratings per person enter the features through two pair-level summaries (section 4). Two near-duplicate ratings would give that taste twice the weight, so I check for pairs above |r| = 0.8.

In [ ]:
corr = people[INTERESTS_ALL].corr()
high = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(INTERESTS_ALL) for b in INTERESTS_ALL[i + 1:]
        if abs(corr.loc[a, b]) > 0.8]
print("interest pairs with |r| > 0.8:", high)
INTERESTS = [c for c in INTERESTS_ALL if c not in {b for _, b, _ in high}]
print(f"kept {len(INTERESTS)} interests, dropped {sorted(set(INTERESTS_ALL) - set(INTERESTS))}")

### 2.4 Preference points: one scale for everyone
The "what I look for" block asks people to split 100 points over six attributes. Waves 6-9 originally rated each item 1-10; the published file already converts those to shares (hence the decimals), but some people's points still do not add up to 100.

In [ ]:
pref_cols = [f"{a}1_1" for a in ATTRS6]
totals = people[pref_cols].sum(axis=1, min_count=6)
pd.DataFrame({
    "people": people.groupby("wave").size(),
    "share with decimals": people.groupby("wave")[pref_cols].apply(lambda d: (d.dropna() % 1 != 0).any(axis=1).mean()),
    "share not summing to 100": (totals.sub(100).abs() > 1).groupby(people["wave"]).mean(),
    "min total": totals.groupby(people["wave"]).min(),
    "max total": totals.groupby(people["wave"]).max(),
}).round(2).T

**Decision.** Rescale each person's six weights so they sum to exactly 100 (section 4), and keep the decimals (rounding to integers would distort waves 6-9 the most).

## 3. One row per woman-man pair
Join each woman's record of a date with the man's record of the same date. The checks fail loudly if anything does not line up.

In [ ]:
SIGNUP_COLS = (["age", "field_cd", "goal", "date", "go_out", "exphappy", "imprace"] + INTERESTS
               + [f"{a}1_1" for a in ATTRS6] + [f"{a}2_1" for a in ATTRS6] + [f"{a}3_1" for a in ATTRS5])
SCORECARD_COLS = ["attr", "sinc", "intel", "fun", "amb", "shar", "like"]   # filled in DURING the date


def build_pair_table(raw: pd.DataFrame) -> pd.DataFrame:
    """One row per woman-man pair: keys, both decisions, the target, both sign-up surveys
    (NaN where unanswered) and, set aside with prefix card_, both scorecards."""
    rec = raw.dropna(subset=["pid"]).astype({"pid": "int64"})
    keep = ["iid", "pid", "wave", "round", "samerace", "dec", "match"] + SIGNUP_COLS + SCORECARD_COLS
    women = rec.loc[rec["gender"] == 0, keep]
    men = rec.loc[rec["gender"] == 1, keep]
    p = women.merge(men, left_on=["iid", "pid"], right_on=["pid", "iid"], suffixes=("_W", "_M"),
                    validate="one_to_one")

    for col in ["wave", "samerace", "match"]:
        assert (p[f"{col}_W"] == p[f"{col}_M"]).all(), f"{col} differs between the two records of a date"
    assert (p["match_W"] == p["dec_W"] * p["dec_M"]).all(), "match is not her yes x his yes"
    assert len(p) == len(women) == len(men), "some records have no mirror record"

    out = pd.DataFrame({
        "iid_W": p["iid_W"], "iid_M": p["iid_M"], "wave": p["wave_W"],
        "event_size": p["round_W"],                 # partners each woman meets = men at the event ("round" differs by side)
        "samerace": p["samerace_W"],
        "dec_W": p["dec_W"], "dec_M": p["dec_M"], "match": p["match_W"],
    })
    for s in ("W", "M"):
        for col in SIGNUP_COLS:
            out[f"{col}_{s}"] = p[f"{col}_{s}"].astype(float)
        for col in SCORECARD_COLS:
            out[f"card_{col}_{s}"] = p[f"{col}_{s}"].astype(float)   # card_attr_W = how attractive SHE found HIM
        out[f"survey_missing_{s}"] = p[f"attr1_1_{s}"].isna().astype(int)
    return out.reset_index(drop=True)


pairs = build_pair_table(raw)
y_all = pairs["match"].to_numpy()
summary["data"] = {
    "date_records": int(len(raw)), "pairs": int(len(pairs)),
    "people": int(pairs["iid_W"].nunique() + pairs["iid_M"].nunique()),
    "women": int(pairs["iid_W"].nunique()), "men": int(pairs["iid_M"].nunique()),
    "events": int(pairs["wave"].nunique()), "matches": int(y_all.sum()), "match_rate": float(y_all.mean()),
    "yes_rate_women": float(pairs["dec_W"].mean()), "yes_rate_men": float(pairs["dec_M"].mean()),
}
log(f"{len(raw):,} date records -> {len(pairs):,} woman-man pairs, {summary['data']['people']} people, "
    f"{summary['data']['events']} events")
log(f"mutual matches: {y_all.sum()} ({y_all.mean():.1%}); she says yes {pairs['dec_W'].mean():.1%}, "
    f"he says yes {pairs['dec_M'].mean():.1%}")

Only about 1 pair in 6 matches, so accuracy is a poor yardstick (predicting "no match" for everyone is 83% accurate). I evaluate with ROC AUC, log loss and a business cost instead (section 8).

## 4. Features (all known before the night)
Three groups, so each can be switched off in the ablation (section 6):
- **profile**: one person's answers (age, goal, self-ratings, ...)
- **pair**: how the two answers relate (age gap, shared interests, whether he has what she says she wants, ...)
- **event**: how many people attend that night

In [ ]:
SIDES = ("W", "M")


def preference_weights(pairs: pd.DataFrame, who: str, block: str = "1_1") -> pd.DataFrame:
    """The six 'what matters' answers, rescaled so each person's weights sum to 100 (section 2.4)."""
    w = pairs[[f"{a}{block}_{who}" for a in ATTRS6]].astype(float)
    w = w.div(w.sum(axis=1, min_count=6).replace(0, np.nan), axis=0) * 100
    w.columns = ATTRS6
    return w


def row_corr(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    den = np.sqrt((a * a).sum(axis=1) * (b * b).sum(axis=1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, (a * b).sum(axis=1) / den, np.nan)


def engineer_features(pairs: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=pairs.index)
    other = {"W": "M", "M": "W"}

    # ---------------- profile: one person at a time ----------------
    for s in SIDES:
        X[f"age_{s}"] = pairs[f"age_{s}"]
        X[f"date_freq_{s}"] = pairs[f"date_{s}"]          # 1 = several times a week ... 7 = almost never
        X[f"go_out_{s}"] = pairs[f"go_out_{s}"]           # same 1-7 scale, any going out
        X[f"exp_happy_{s}"] = pairs[f"exphappy_{s}"]      # expected happiness with the night, 1-10
        goal = pairs[f"goal_{s}"]                         # a category, not a scale: one-hot the three intents that differ
        X[f"goal_serious_{s}"] = (goal == 4).astype(float).where(goal.notna())
        X[f"goal_get_date_{s}"] = (goal == 3).astype(float).where(goal.notna())
        X[f"goal_fun_night_{s}"] = (goal == 1).astype(float).where(goal.notna())
        for a in ATTRS5:
            X[f"self_{a}_{s}"] = pairs[f"{a}3_1_{s}"]     # self-rating 1-10
        X[f"wants_looks_{s}"] = preference_weights(pairs, s)["attr"]
        X[f"survey_missing_{s}"] = pairs[f"survey_missing_{s}"].astype(float)

    # ---------------- pair: compatibility ----------------
    iw = pairs[[f"{i}_W" for i in INTERESTS]].to_numpy(dtype=float)
    im = pairs[[f"{i}_M" for i in INTERESTS]].to_numpy(dtype=float)
    X["interest_corr"] = row_corr(iw, im)                       # same shape of interests?
    X["interest_gap"] = np.abs(iw - im).mean(axis=1)            # same intensity?
    X["age_gap"] = pairs["age_M"] - pairs["age_W"]              # positive = he is older
    X["age_gap_abs"] = X["age_gap"].abs()
    X["same_field"] = (pairs["field_cd_W"] == pairs["field_cd_M"]).astype(float).where(
        pairs["field_cd_W"].notna() & pairs["field_cd_M"].notna())
    X["looks_gap"] = pairs["attr3_1_M"] - pairs["attr3_1_W"]    # self-rated looks, him minus her

    # "Does the partner have what I say I want?" My weights on the five rateable
    # attributes x the partner's self-ratings (1-10 scale).
    for s in SIDES:
        w = preference_weights(pairs, s)[ATTRS5]
        w = w.div(w.sum(axis=1), axis=0)
        partner_self = pairs[[f"{a}3_1_{other[s]}" for a in ATTRS5]].to_numpy(dtype=float)
        X[f"pref_fit_{s}"] = (w.to_numpy() * partner_self).sum(axis=1)
        X.loc[w.isna().any(axis=1) | np.isnan(partner_self).any(axis=1), f"pref_fit_{s}"] = np.nan

    # Same race only matters to people who say it matters: interaction term.
    X["same_race"] = pairs["samerace"].astype(float)
    for s in SIDES:
        X[f"race_mismatch_{s}"] = (1 - pairs["samerace"]) * pairs[f"imprace_{s}"]

    # ---------------- event ----------------
    X["event_size"] = pairs["event_size"].astype(float)
    return X


FEATURE_GROUPS = {
    "profile": lambda c: c.startswith(("age_W", "age_M", "date_freq", "go_out", "exp_happy", "goal_", "self_",
                                       "wants_looks", "survey_missing")),
    "pair": lambda c: c.startswith(("interest_", "age_gap", "same_field", "looks_gap", "pref_fit", "same_race",
                                    "race_mismatch")),
    "event": lambda c: c == "event_size",
}


def group_of(col: str) -> str:
    return next(g for g, test in FEATURE_GROUPS.items() if test(col))


FEATURE_INFO = {
    "age": "Age in years",
    "date_freq": "How often they go on dates (1 = several times a week, 7 = almost never)",
    "go_out": "How often they go out at all (1 = several times a week, 7 = almost never)",
    "exp_happy": "Expected happiness with the people they will meet (1-10)",
    "goal_serious": "Main goal: a serious relationship",
    "goal_get_date": "Main goal: to get a date",
    "goal_fun_night": "Main goal: a fun night out",
    "self_attr": "Self-rated attractiveness (1-10)",
    "self_sinc": "Self-rated sincerity (1-10)",
    "self_intel": "Self-rated intelligence (1-10)",
    "self_fun": "Self-rated fun (1-10)",
    "self_amb": "Self-rated ambition (1-10)",
    "wants_looks": "Share of the 100 preference points given to 'attractive'",
    "survey_missing": "Did not fill in the sign-up survey (answers imputed)",
    "interest_corr": "Correlation of the two people's interest ratings",
    "interest_gap": "Mean absolute difference of the interest ratings",
    "age_gap": "His age minus her age",
    "age_gap_abs": "Absolute age difference",
    "same_field": "Same field of study",
    "looks_gap": "His self-rated looks minus hers",
    "pref_fit": "Partner's self-ratings weighted by what this person says matters (1-10)",
    "same_race": "Same race/ethnicity",
    "race_mismatch": "Different race x how much this person says same race matters (0-10)",
    "event_size": "Number of men at the event (partners each woman meets)",
}


def describe(col: str) -> str:
    base = col[:-2] if col.endswith(("_W", "_M")) else col
    return FEATURE_INFO.get(base, col) + {"_W": " (woman)", "_M": " (man)"}.get(col[-2:], "")


X_all = engineer_features(pairs)
FEATURES = list(X_all.columns)
log(f"{len(FEATURES)} features: " + ", ".join(f"{g}={sum(group_of(c) == g for c in FEATURES)}" for g in FEATURE_GROUPS))
pd.DataFrame({"group": [group_of(c) for c in FEATURES], "description": [describe(c) for c in FEATURES],
              "missing": X_all.isna().mean().round(3).to_numpy()}, index=FEATURES)

Feature blocks I also tried on top of this set. Section 6 scores them the same way as the final set, and none earned a place.

In [ ]:
FIELD_GROUPS = {  # 18 field codes -> a few broad groups (the rest = "other")
    1: "law", 2: "stem", 3: "social", 4: "medicine", 5: "stem", 6: "arts_humanities",
    7: "arts_humanities", 8: "business", 9: "social", 10: "stem", 11: "social", 13: "social",
    14: "arts_humanities", 15: "arts_humanities", 16: "arts_humanities", 17: "arts_humanities",
}


def candidate_additions(pairs: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    out["raw interest levels (per person)"] = pd.DataFrame(
        {f"int_{i}_{s}": pairs[f"{i}_{s}"] for s in SIDES for i in INTERESTS}, index=pairs.index)
    out["all six preference weights per person"] = pd.concat(
        [preference_weights(pairs, s).drop(columns="attr").add_prefix("pref_").add_suffix(f"_{s}") for s in SIDES],
        axis=1)
    out["field of study, 6 groups per person"] = pd.DataFrame(
        {f"field_{g}_{s}": (pairs[f"field_cd_{s}"].map(FIELD_GROUPS) == g).astype(float)
         .where(pairs[f"field_cd_{s}"].notna())
         for s in SIDES for g in sorted(set(FIELD_GROUPS.values()))}, index=pairs.index)
    out["beliefs about what the other side wants"] = pd.concat(
        [preference_weights(pairs, s, block="2_1").add_prefix("belief_").add_suffix(f"_{s}") for s in SIDES], axis=1)
    return out

## 5. Split by event, in date order
The organiser scores the pairs of an *upcoming* event using what it learned from *past* events. So whole events go to one side of the split, in time order: waves 1-13 train, 14-16 validation, 17-21 test. The test events are used once, at the end.

In [ ]:
wave = pairs["wave"].to_numpy()
is_train, is_val, is_test = wave <= 13, (wave >= 14) & (wave <= 16), wave >= 17
is_dev = is_train | is_val          # train + validation = development events, used for the final fit
SPLITS = [("train", is_train), ("validation", is_val), ("test", is_test)]
for name, m in SPLITS:
    log(f"{name:>10}: waves {wave[m].min()}-{wave[m].max()}, {m.sum():,} pairs, match rate {y_all[m].mean():.3f}")
summary["split"] = {n: {"waves": sorted(int(w) for w in set(wave[m])), "pairs": int(m.sum()),
                        "matches": int(y_all[m].sum()), "match_rate": float(y_all[m].mean())} for n, m in SPLITS}

by_wave = pairs.groupby("wave").agg(match=("match", "mean"), her=("dec_W", "mean"), his=("dec_M", "mean"))
split_of = np.where(by_wave.index >= 17, "test", np.where(by_wave.index >= 14, "validation", "train"))
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.bar(by_wave.index, by_wave["match"], color=[{"train": "#2a6fdb", "validation": "#e8833a", "test": "#2ca58d"}[s]
                                               for s in split_of])
ax.plot(by_wave.index, by_wave["her"], "o-", color="#d1495b", label="her yes rate")
ax.plot(by_wave.index, by_wave["his"], "o-", color="#555", label="his yes rate")
ax.set(xlabel="wave (event, in date order); bars = mutual match rate, blue train / orange validation / green test",
       ylabel="rate", title="The target varies a lot between nights", xticks=by_wave.index)
ax.set_ylim(0, 0.8)
ax.legend(frameon=False, ncol=2, loc="upper left")
fig.tight_layout()
fig.savefig(FIG / "target_by_wave.png")
plt.show()

### What a random split would have hidden
Same model, development events only, 5-fold cross-validation two ways: shuffled rows versus whole events. With shuffled rows the same people sit in both training and evaluation folds, so the model can recognise them. The event folds are fixed (events in date order, dealt round-robin into 5 folds) so the result does not depend on the scikit-learn version.

In [ ]:
X_dev, y_dev, g_dev = X_all[is_dev], y_all[is_dev], wave[is_dev]


def preprocessor() -> Pipeline:
    """Median imputation + standardisation, always fitted on training rows only."""
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])


def sk_model(C: float) -> LogisticRegression:
    return LogisticRegression(C=C, max_iter=10_000, tol=1e-8)


def cv_auc(splitter, groups=None, C=0.003) -> list[float]:
    out = []
    for tr, te in splitter.split(X_dev, y_dev, groups):
        m = Pipeline([("pre", preprocessor()), ("lr", sk_model(C))]).fit(X_dev.iloc[tr], y_dev[tr])
        out.append(roc_auc_score(y_dev[te], m.predict_proba(X_dev.iloc[te])[:, 1]))
    return out


rows_cv = cv_auc(KFold(5, shuffle=True, random_state=SEED))
dev_waves = np.unique(g_dev)
event_fold = pd.Series(np.arange(len(dev_waves)) % 5, index=dev_waves)
event_cv = cv_auc(PredefinedSplit(event_fold.reindex(g_dev).to_numpy()))
seen = []   # share of evaluation pairs involving someone already seen in training (random split)
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X_dev):
    d_tr, d_te = pairs[is_dev].iloc[tr], pairs[is_dev].iloc[te]
    seen.append(np.mean(d_te["iid_W"].isin(d_tr["iid_W"]) | d_te["iid_M"].isin(d_tr["iid_M"])))
summary["leakage_check"] = {
    "random_rows_auc_mean": float(np.mean(rows_cv)), "random_rows_auc_sd": float(np.std(rows_cv)),
    "by_event_auc_mean": float(np.mean(event_cv)), "by_event_auc_sd": float(np.std(event_cv)),
    "share_eval_pairs_with_person_seen_in_training": float(np.mean(seen)),
}
log(f"5-fold CV AUC, random rows: {np.mean(rows_cv):.3f} | whole events: {np.mean(event_cv):.3f} "
    f"({np.mean(seen):.0%} of random-split evaluation pairs involve someone seen in training)")

The random split looks better only because it lets the model meet the same people twice. Every evaluation below holds out whole events.

## 6. Regularisation strength C
C is the one hyperparameter of the model itself. I choose it by leave-one-event-out over the 16 development events, scored by **log loss**: the decision threshold in section 8 is derived from the probabilities, so they need to be calibrated, not just well ranked. With only three validation events a single split is too noisy to choose from.

In [ ]:
C_GRID = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 1.0]


def lowo_scores(cols: list[str], C: float, X: pd.DataFrame | None = None) -> dict:
    """Leave-one-event-out predictions on the development events, then score them."""
    X = X_dev[cols] if X is None else X
    oof = np.zeros(len(y_dev))
    for tr, te in LeaveOneGroupOut().split(X, y_dev, g_dev):
        m = Pipeline([("pre", preprocessor()), ("lr", sk_model(C))]).fit(X.iloc[tr], y_dev[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    per_event = [roc_auc_score(y_dev[g_dev == g], oof[g_dev == g]) for g in np.unique(g_dev)]
    return {"log_loss": log_loss(y_dev, oof), "auc": roc_auc_score(y_dev, oof), "mean_event_auc": float(np.mean(per_event))}


c_search = pd.DataFrame([{"C": C, **lowo_scores(FEATURES, C)} for C in C_GRID])
BEST_C = float(c_search.loc[c_search["log_loss"].idxmin(), "C"])
summary["c_search"] = c_search.to_dict("records")
summary["best_C"] = BEST_C
log(f"chosen C = {BEST_C}")
c_search.round(4)

### Do the feature groups earn their place?
Drop one group at a time, then add each tried-and-dropped block on top. Same C, same leave-one-event-out scoring.

In [ ]:
ablation = [{"dropped": "nothing (all features)", "n_features": len(FEATURES), **lowo_scores(FEATURES, BEST_C)}]
for g in FEATURE_GROUPS:
    cols = [c for c in FEATURES if group_of(c) != g]
    ablation.append({"dropped": g, "n_features": len(cols), **lowo_scores(cols, BEST_C)})
ablation = pd.DataFrame(ablation)

tried = []
for name, extra in candidate_additions(pairs).items():
    cols = FEATURES + list(extra.columns)
    tried.append({"added": name, "n_features": len(cols),
                  **lowo_scores(cols, BEST_C, X=pd.concat([X_all, extra], axis=1).loc[is_dev, cols])})
tried = pd.DataFrame(tried)
summary["ablation"] = ablation.to_dict("records")
summary["tried_and_dropped"] = tried.to_dict("records")
print("Dropping a group:\n" + ablation.round(4).to_string(index=False))
print("\nAdding a block on top of the final set:\n" + tried.round(4).to_string(index=False))

Dropping the profile or the pair group makes both log loss and AUC worse, so both stay. The event group is a single feature and close to neutral; I keep it because event size is something the organiser controls. None of the added blocks lowers the log loss.

## 7. The same logistic regression, three ways
All three minimise the same objective, so on the same data they should land on the same weights:

$$J(w, b) = \text{mean binary cross-entropy} + \frac{\lambda}{2}\lVert w\rVert^2, \qquad \lambda = \frac{1}{C\,n}$$

scikit-learn writes it as $C\sum \text{loss} + \tfrac12\lVert w\rVert^2$; dividing by $Cn$ gives the line above. The intercept is not penalised (scikit-learn does not penalise it either), which is why I add the L2 term by hand instead of using the optimiser's `weight_decay` (that would also shrink the bias).

In [ ]:
def l2_strength(C: float, n_rows: int) -> float:
    return 1.0 / (C * n_rows)


def objective(X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float, lam: float) -> float:
    """J(w, b) in float64 numpy, used to check that all three reach the same minimum."""
    z = X @ w + b
    return float(np.mean(np.logaddexp(0.0, z) - y * z) + 0.5 * lam * np.dot(w, w))


# ---------- 1) Manual loop: raw tensors, autograd, hand-written update ----------
def train_manual(X: np.ndarray, y: np.ndarray, lam: float, lr: float, epochs: int):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    w = torch.zeros(Xt.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    for _ in range(epochs):
        # forward pass: linear score -> probability -> penalised cross-entropy
        p = torch.clamp(torch.sigmoid(Xt @ w + b), 1e-7, 1 - 1e-7)   # never take log(0)
        bce = -(yt * torch.log(p) + (1 - yt) * torch.log(1 - p)).mean()
        loss = bce + 0.5 * lam * (w ** 2).sum()
        loss.backward()                  # autograd fills w.grad and b.grad
        with torch.no_grad():            # gradient step outside the graph
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()                   # otherwise the next backward() adds on top
        b.grad.zero_()
        history.append(loss.item())
    return {"w": w.detach().clone(), "b": b.detach().clone()}, history


def predict_manual(params: dict, X: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return torch.sigmoid(torch.tensor(X, dtype=torch.float32) @ params["w"] + params["b"]).numpy().astype(float)


# ---------- 2) Standard workflow: nn.Module + loss module + torch.optim ----------
class LogisticRegressionNet(nn.Module):
    """One linear layer. forward() returns logits; the sigmoid lives in BCEWithLogitsLoss (numerically stable)."""

    def __init__(self, n_features: int):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x).squeeze(-1)


def train_module(X: np.ndarray, y: np.ndarray, lam: float, lr: float, epochs: int, seed: int = SEED):
    torch.manual_seed(seed)
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    model = LogisticRegressionNet(Xt.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Cosine decay of the step size to 0 over the run: with a constant rate Adam keeps hovering
    # around the minimum in float32; the decay lets it settle exactly on it.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []
    # Full batch: ~3k rows fit in memory and a fixed batch makes the run deterministic,
    # so any difference to scikit-learn comes from the optimiser alone.
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xt), yt) + 0.5 * lam * (model.linear.weight ** 2).sum()
        loss.backward()
        optimizer.step()
        scheduler.step()
        history.append(loss.item())
    return model, history


def predict_module(model: LogisticRegressionNet, X: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X, dtype=torch.float32))).numpy().astype(float)


def module_params(model: LogisticRegressionNet) -> tuple[np.ndarray, float]:
    return model.linear.weight.detach().numpy().ravel().astype(float), float(model.linear.bias.detach().item())

### Learning rates for the two PyTorch versions
Fit on the training events with $\lambda = 1/(Cn)$, so the objective is scikit-learn's. The learning rate is an optimiser setting, not a model setting: its job is to reach the minimum of the training objective. (For Adam it is the starting rate of a cosine schedule that decays to 0 by the last epoch.) So I pick the rate with the **lowest final training objective that did not diverge**, and report validation log loss alongside. Choosing on validation loss could reward a rate that is too small (stopping early acts as extra regularisation), and I would silently be comparing a different model to scikit-learn.

In [ ]:
EPOCHS = 1500
pre_tr = preprocessor().fit(X_all[is_train])
Xs_tr, Xs_val = pre_tr.transform(X_all[is_train]), pre_tr.transform(X_all[is_val])
y_tr, y_val = y_all[is_train].astype(float), y_all[is_val].astype(float)
lam_tr = l2_strength(BEST_C, len(y_tr))
sk_tr = sk_model(BEST_C).fit(Xs_tr, y_tr)
w_sk_tr = sk_tr.coef_.ravel()
J_sk_tr = objective(Xs_tr, y_tr, w_sk_tr, sk_tr.intercept_[0], lam_tr)


def lr_row(method: str, lr: float, w: np.ndarray, b: float, p_val: np.ndarray, hist: list) -> dict:
    J = objective(Xs_tr, y_tr, w, b, lam_tr)
    return {"method": method, "lr": lr, "train_objective_gap": J - J_sk_tr,
            "val_log_loss": log_loss(y_val, np.clip(p_val, 1e-7, 1 - 1e-7)),
            "max_coef_diff_vs_sklearn": float(np.abs(w - w_sk_tr).max()),
            "diverged": bool(not np.isfinite(J) or J > hist[0])}


lr_search = []
for lr in [0.01, 0.1, 0.5, 1.0, 2.0, 4.0]:
    params, hist = train_manual(Xs_tr, y_tr, lam_tr, lr, EPOCHS)
    lr_search.append(lr_row("manual", lr, params["w"].numpy().astype(float), float(params["b"].item()),
                            predict_manual(params, Xs_val), hist))
for lr in [0.001, 0.01, 0.05, 0.1]:
    model, hist = train_module(Xs_tr, y_tr, lam_tr, lr, EPOCHS)
    lr_search.append(lr_row("module", lr, *module_params(model), predict_module(model, Xs_val), hist))
lr_search = pd.DataFrame(lr_search)
ok = lr_search[~lr_search["diverged"]]
BEST_LR = {m: float(ok[ok["method"] == m].sort_values("train_objective_gap")["lr"].iloc[0]) for m in ("manual", "module")}
summary.update(lr_search=lr_search.to_dict("records"), best_lr=BEST_LR, epochs=EPOCHS)
log(f"chosen learning rates: {BEST_LR}")
lr_search

### Final fit on all 16 development events
Same rows, same preprocessing (fitted on the development events), same objective, same C.

In [ ]:
pre = preprocessor().fit(X_all[is_dev])
Xs_dev, Xs_test = pre.transform(X_all[is_dev]), pre.transform(X_all[is_test])
y_dev_f, y_test = y_dev.astype(float), y_all[is_test]
lam = l2_strength(BEST_C, len(y_dev_f))

fit_time = {}
t = time.time()
sk = sk_model(BEST_C).fit(Xs_dev, y_dev_f)
fit_time["sklearn"] = time.time() - t
t = time.time()
manual_params, manual_hist = train_manual(Xs_dev, y_dev_f, lam, BEST_LR["manual"], EPOCHS)
fit_time["manual"] = time.time() - t
t = time.time()
module, module_hist = train_module(Xs_dev, y_dev_f, lam, BEST_LR["module"], EPOCHS)
fit_time["module"] = time.time() - t

coefs = pd.DataFrame({
    "feature": FEATURES, "group": [group_of(c) for c in FEATURES], "description": [describe(c) for c in FEATURES],
    "sklearn": sk.coef_.ravel(), "manual": manual_params["w"].numpy().astype(float), "module": module_params(module)[0],
})
intercepts = {"sklearn": float(sk.intercept_[0]), "manual": float(manual_params["b"].item()),
              "module": module_params(module)[1]}
J_final = {k: objective(Xs_dev, y_dev_f, coefs[k].to_numpy(), intercepts[k], lam) for k in LABEL}
log("final training objective: " + ", ".join(f"{k}={v:.8f}" for k, v in J_final.items()))
log("fit time (s): " + ", ".join(f"{k}={v:.2f}" for k, v in fit_time.items()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
lim = np.abs(coefs[list(LABEL)].to_numpy()).max() * 1.1
ax[0].plot([-lim, lim], [-lim, lim], color="#999", lw=0.8, ls="--")
ax[0].scatter(coefs["sklearn"], coefs["manual"], s=22, color=C3["manual"], label=LABEL["manual"])
ax[0].scatter(coefs["sklearn"], coefs["module"], s=8, color=C3["module"], label=LABEL["module"])
ax[0].set(xlabel="scikit-learn coefficient", ylabel="PyTorch coefficient", title="Same weights, three implementations")
ax[0].legend(frameon=False, fontsize=7)
for k, h in [("manual", manual_hist), ("module", module_hist)]:
    ax[1].plot(np.maximum(np.array(h) - J_final["sklearn"], 1e-9), color=C3[k], label=LABEL[k])
ax[1].set(yscale="log", xlabel="epoch", ylabel="objective minus scikit-learn's minimum",
          title="Both PyTorch versions converge to the same minimum")
ax[1].legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "three_implementations.png")
plt.show()

## 8. Test events: three implementations vs naive baselines

**Business framing.** A seated pair that does not match wastes a 4-minute slot for two people (**EUR 5**, a false positive). A match we never seat is a lost customer outcome (**EUR 20**, a false negative). Seat a pair when $p \cdot c_{fn} > (1-p) \cdot c_{fp}$, i.e. when $p > c_{fp}/(c_{fp}+c_{fn}) = 0.2$. This rule only works if the probabilities are calibrated, which is why C was tuned on log loss.

In [ ]:
COST_FP = 5.0    # EUR: a scheduled date that ends without a mutual yes
COST_FN = 20.0   # EUR: a pair that would have matched but was never seated together


def cost_threshold(c_fp: float = COST_FP, c_fn: float = COST_FN) -> float:
    return c_fp / (c_fp + c_fn)


def confusion(y: np.ndarray, p: np.ndarray, t: float) -> dict:
    pred, y = p >= t, y.astype(bool)
    return {"tp": int((pred & y).sum()), "fp": int((pred & ~y).sum()),
            "fn": int((~pred & y).sum()), "tn": int((~pred & ~y).sum())}


def business_cost(cm: dict, c_fp: float = COST_FP, c_fn: float = COST_FN) -> float:
    return cm["fp"] * c_fp + cm["fn"] * c_fn


def evaluate(y: np.ndarray, p: np.ndarray, t: float, c_fp: float = COST_FP, c_fn: float = COST_FN) -> dict:
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    cm = confusion(y, p, t)
    flagged = cm["tp"] + cm["fp"]
    return {
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p), "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier": brier_score_loss(y, p), "accuracy": (cm["tp"] + cm["tn"]) / len(y),
        "precision": cm["tp"] / flagged if flagged else np.nan,
        "recall": cm["tp"] / (cm["tp"] + cm["fn"]) if (cm["tp"] + cm["fn"]) else np.nan,
        "share_scheduled": flagged / len(y), "cost_eur": business_cost(cm, c_fp, c_fn), **cm,
    }


def cost_curve(y: np.ndarray, p: np.ndarray, c_fp: float, c_fn: float, thresholds=None) -> pd.DataFrame:
    thresholds = np.linspace(0, 1, 201) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": t, "cost_eur": business_cost(confusion(y, p, t), c_fp, c_fn),
                          **confusion(y, p, t)} for t in thresholds])


def calibration_table(y: np.ndarray, p: np.ndarray, n_bins: int = 8) -> pd.DataFrame:
    """Pairs sorted into equal-size bins of predicted probability: mean prediction vs observed match rate."""
    df = pd.DataFrame({"y": y, "p": p})
    df["bin"] = pd.qcut(df["p"].rank(method="first"), n_bins, labels=False)
    g = df.groupby("bin").agg(mean_pred=("p", "mean"), match_rate=("y", "mean"), n=("y", "size"))
    g["se"] = np.sqrt(g["match_rate"] * (1 - g["match_rate"]) / g["n"])
    return g.reset_index()


def mean_auc_within(df: pd.DataFrame, p: np.ndarray, key: str) -> float:
    """Average AUC inside each group (event, woman or man). Inside one woman's list it asks: does the
    model rank the men she said yes to (and who said yes back) above the others?"""
    return float(np.mean([roc_auc_score(g["match"], g["p"]) for _, g in df.assign(p=p).groupby(key)
                          if 0 < g["match"].mean() < 1]))


def bootstrap_ci(y: np.ndarray, preds: dict, n_boot: int = 1000, seed: int = SEED) -> dict:
    """Pair-level bootstrap of test AUC. Pairs of the same person are not independent,
    so these intervals are on the narrow (optimistic) side."""
    rng = np.random.default_rng(seed)
    aucs = {k: [] for k in preds}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        for k, v in preds.items():
            aucs[k].append(roc_auc_score(y[idx], v[idx]))
    return {"auc_ci": {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in aucs.items()}}


T_STAR = cost_threshold()
base_rate = float(y_dev.mean())
test_pairs = pairs[is_test].reset_index(drop=True)
preds = {"sklearn": sk.predict_proba(Xs_test)[:, 1], "manual": predict_manual(manual_params, Xs_test),
         "module": predict_module(module, Xs_test)}
results = {name: evaluate(y_test, p, T_STAR) for name, p in preds.items()}
results["baseline: base rate (schedule nobody)"] = evaluate(y_test, np.full(len(y_test), base_rate), T_STAR)
results["baseline: schedule everyone"] = evaluate(y_test, np.ones(len(y_test)), T_STAR)
results_df = pd.DataFrame(results).T
for name, p in preds.items():
    for col, key in [("mean_event_auc", "wave"), ("within_woman_auc", "iid_W"), ("within_man_auc", "iid_M")]:
        results_df.loc[name, col] = mean_auc_within(test_pairs, p, key)

agreement = {}
for i, a in enumerate(LABEL):
    for b in list(LABEL)[i + 1:]:
        agreement[f"{a} vs {b}"] = {
            "max_abs_coef_diff": float(np.abs(coefs[a] - coefs[b]).max()),
            "intercept_diff": float(abs(intercepts[a] - intercepts[b])),
            "max_abs_prob_diff_test": float(np.abs(preds[a] - preds[b]).max()),
            "same_decision_share_test": float(np.mean((preds[a] >= T_STAR) == (preds[b] >= T_STAR))),
        }
ci = bootstrap_ci(y_test, preds)
summary.update(
    test={k: {kk: (None if pd.isna(vv) else float(vv)) for kk, vv in v.items()} for k, v in results_df.to_dict("index").items()},
    agreement=agreement, objective=J_final, intercepts=intercepts, bootstrap=ci, fit_seconds=fit_time,
    threshold={"cost_fp": COST_FP, "cost_fn": COST_FN, "t_star": T_STAR, "dev_base_rate": base_rate},
)
log(f"bootstrap 95% CI of test AUC (scikit-learn): {ci['auc_ci']['sklearn'][0]:.3f} - {ci['auc_ci']['sklearn'][1]:.3f}")
print(pd.DataFrame(agreement).T.to_string(), "\n")
results_df[["roc_auc", "mean_event_auc", "within_woman_auc", "within_man_auc", "pr_auc", "log_loss", "brier",
            "precision", "recall", "share_scheduled", "cost_eur"]].astype(float).round(4)

### Is the threshold sensible, and are the probabilities calibrated?
The cost rule assumes calibrated probabilities. Two checks: where the cost is actually lowest on the validation events (train-only model, test events untouched), and predicted versus observed match rate on the test events.

In [ ]:
val_curve = cost_curve(y_val, sk_tr.predict_proba(Xs_val)[:, 1], COST_FP, COST_FN)
best_val_t = float(val_curve.loc[val_curve["cost_eur"].idxmin(), "threshold"])
summary["threshold"]["validation_cost_minimising_t"] = best_val_t
summary["calibration_test"] = {k: calibration_table(y_test, p).to_dict("records") for k, p in preds.items()}
log(f"theory threshold {T_STAR:.3f}, validation cost minimum at {best_val_t:.3f}")

STYLE = {"sklearn": dict(lw=3.5, ls="-"), "manual": dict(lw=1.8, ls="--"), "module": dict(lw=1.0, ls=":")}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, p in preds.items():
    cal = calibration_table(y_test, p)
    ax[0].plot(cal["mean_pred"], cal["match_rate"], marker="o", color=C3[k], label=LABEL[k], **STYLE[k])
    cc = cost_curve(y_test, p, COST_FP, COST_FN)
    ax[1].plot(cc["threshold"], cc["cost_eur"], color=C3[k], label=LABEL[k], **STYLE[k])
ax[0].plot([0, 0.35], [0, 0.35], color="#999", lw=0.8, ls="--")
ax[0].set(xlim=(0, 0.35), ylim=(0, 0.35), xlabel="predicted match probability (bin mean)",
          ylabel="observed match rate", title="Predicted vs actual, test events (8 bins)")
ax[1].axhline(results["baseline: schedule everyone"]["cost_eur"], color="#999", ls=":", label="schedule everyone")
ax[1].axhline(results["baseline: base rate (schedule nobody)"]["cost_eur"], color="#666", ls="--", label="schedule nobody")
ax[1].axvline(T_STAR, color="#333", lw=0.8)
ax[1].set(xlim=(0, 0.5), xlabel="decision threshold", ylabel="cost on test events (EUR)",
          title=f"Business cost (FP EUR {COST_FP:.0f}, FN EUR {COST_FN:.0f}); vertical line = 0.2")
for a in ax:
    a.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "calibration_and_cost.png")
plt.show()

## 9a. Event planner: the decision the model is for
Suppose the night is cut short so each person only meets a share of the other side. The model's plan picks the set of dates with the highest total predicted probability, with the same number of dates per person. That is a small linear program (bipartite b-matching); its constraint matrix is totally unimodular, so the optimum is already 0/1 and no rounding is needed. I compare the mutual matches it keeps on the test events with random plans of the same size and with a perfect plan.

In [ ]:
def seating_plan(event: pd.DataFrame, score: np.ndarray, share: float) -> np.ndarray:
    """Boolean mask over the event's rows: the dates with the highest total score, at most k per person."""
    women = pd.factorize(event["iid_W"])[0]
    men = pd.factorize(event["iid_M"])[0]
    n, n_w, n_m = len(event), women.max() + 1, men.max() + 1
    A = coo_matrix((np.ones(2 * n), (np.concatenate([women, n_w + men]), np.concatenate([np.arange(n), np.arange(n)]))),
                   shape=(n_w + n_m, n))
    b = np.concatenate([np.full(n_w, int(np.ceil(share * n_m))), np.full(n_m, int(np.ceil(share * n_w)))])
    res = linprog(-np.asarray(score, dtype=float), A_ub=A, b_ub=b, bounds=(0, 1), method="highs")
    return res.x > 0.5


def plan_events(pairs: pd.DataFrame, p: np.ndarray, share: float = 0.5, n_random: int = 100, seed: int = SEED) -> pd.DataFrame:
    """Per event: matches kept by the model's plan, by random plans of the same size, and by a perfect plan."""
    rng = np.random.default_rng(seed)
    rows = []
    for w, ev in pairs.assign(_p=p).groupby("wave"):
        y = ev["match"].to_numpy()
        chosen = seating_plan(ev, ev["_p"].to_numpy(), share)
        rand = [int(y[seating_plan(ev, rng.random(len(ev)), share)].sum()) for _ in range(n_random)]
        perfect = y[seating_plan(ev, y + rng.random(len(ev)) * 1e-3, share)].sum()
        rows.append({"wave": int(w), "pairs": len(ev), "all_matches": int(y.sum()), "dates_held": int(chosen.sum()),
                     "model_plan": int(y[chosen].sum()), "random_plan": float(np.mean(rand)),
                     "perfect_plan": int(perfect), "random_draws": rand})
    return pd.DataFrame(rows)


planner = {}
for share in (0.3, 0.5, 0.7):
    pl = plan_events(test_pairs, preds["sklearn"], share=share, n_random=200)
    rand_tot = np.array(pl["random_draws"].tolist()).sum(axis=0)   # total over events, per random draw
    planner[str(share)] = {
        "all_matches": int(pl["all_matches"].sum()), "dates_held": int(pl["dates_held"].sum()),
        "model_plan": int(pl["model_plan"].sum()), "random_plan": float(rand_tot.mean()),
        "random_plan_sd": float(rand_tot.std()), "perfect_plan": int(pl["perfect_plan"].sum()),
        "share_random_draws_at_least_as_good": float(np.mean(rand_tot >= pl["model_plan"].sum())),
    }
    if share == 0.5:
        plan_half = pl.drop(columns="random_draws")
summary["planner"] = {"by_share": planner, "by_event_half": plan_half.to_dict("records")}

fig, ax = plt.subplots(figsize=(7, 3.3))
xs = np.arange(len(plan_half))
ax.bar(xs - 0.27, plan_half["random_plan"], 0.27, color="#bbb", label="random plan (mean)")
ax.bar(xs, plan_half["model_plan"], 0.27, color="#2a6fdb", label="model plan")
ax.bar(xs + 0.27, plan_half["perfect_plan"], 0.27, color="#2ca58d", label="perfect plan")
ax.set_xticks(xs, [f"wave {w}" for w in plan_half["wave"]])
ax.set(ylabel="mutual matches kept", title="Everyone meets half the room")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "planner.png")
plt.show()
pd.DataFrame(planner).T.round(2)

## 9b. The missing ingredient: physical attraction
None of the deployable features describe how someone looks. How much would that be worth? Same model (C re-tuned per variant with the same leave-one-event-out search), trained on the development events, scored on the test events, adding information step by step:

1. deployable model (sign-up data only)
2. \+ a **looks reputation**: the average attractiveness score each person got from everyone *else* they met that night, a stand-in for a crowd-rated photo
3. \+ how attractive the two people found *each other* (scorecard)
4. \+ the full scorecard (sincere, intelligent, fun, ambitious, shared interests, overall like)

Steps 2-4 come from the night itself and are **not** available before the event. They measure the size of the gap.

In [ ]:
def looks_reputation(raw: pd.DataFrame, pairs: pd.DataFrame) -> pd.DataFrame:
    """Leave-one-out mean of the attractiveness ratings a person received from everyone else that night."""
    r = raw.dropna(subset=["pid"]).astype({"pid": "int64"})[["iid", "pid", "attr_o"]]
    totals = r.groupby("iid")["attr_o"].agg(["sum", "count"])
    received = r.set_index(["iid", "pid"])["attr_o"]

    def loo(person: pd.Series, partner: pd.Series) -> np.ndarray:
        own = received.reindex(pd.MultiIndex.from_arrays([person, partner])).to_numpy()
        has_own = ~np.isnan(own)
        num = totals["sum"].reindex(person).to_numpy() - np.where(has_own, own, 0.0)
        den = totals["count"].reindex(person).to_numpy().astype(float) - has_own
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(den > 0, num / den, np.nan)

    return pd.DataFrame({"looks_rep_W": loo(pairs["iid_W"], pairs["iid_M"]),   # how men rated her
                         "looks_rep_M": loo(pairs["iid_M"], pairs["iid_W"])},  # how women rated him
                        index=pairs.index)


X_ext = pd.concat([X_all, looks_reputation(raw, pairs), pairs.filter(like="card_")], axis=1)
rep_cols = ["looks_rep_W", "looks_rep_M"]
variants = {
    "1. sign-up data only (deployable)": FEATURES,
    "2. + looks reputation from other partners": FEATURES + rep_cols,
    "3. + how attractive they found each other": FEATURES + rep_cols + ["card_attr_W", "card_attr_M"],
    "4. + the full scorecard": FEATURES + rep_cols + [f"card_{a}_{s}" for s in SIDES for a in SCORECARD_COLS],
}
whatif = []
for name, cols in variants.items():
    best = min(C_GRID, key=lambda C: lowo_scores(cols, C, X=X_ext.loc[is_dev, cols])["log_loss"])
    m = Pipeline([("pre", preprocessor()), ("lr", sk_model(best))]).fit(X_ext.loc[is_dev, cols], y_dev)
    p = m.predict_proba(X_ext.loc[is_test, cols])[:, 1]
    r = evaluate(y_test, p, T_STAR)
    pl = plan_events(test_pairs, p, share=0.5, n_random=1)
    whatif.append({"information": name, "C": best, "roc_auc": r["roc_auc"], "pr_auc": r["pr_auc"],
                   "log_loss": r["log_loss"], "cost_eur": r["cost_eur"], "planner_matches": int(pl["model_plan"].sum())})
whatif = pd.DataFrame(whatif)
whatif.round(3)

Why is attraction so hard to get before the night? The only looks signal on the sign-up form is a self-rating. Below: how well it tracks the ratings people actually received, and whose half of the match the sign-up form can predict (same features and C, target = one person's yes).

In [ ]:
per_person = raw.groupby("iid").agg(gender=("gender", "first"), self_attr=("attr3_1", "first"),
                                    looks_received=("attr_o", "mean"), yes_received=("dec_o", "mean"))
corr_self = {"all": float(per_person["self_attr"].corr(per_person["looks_received"]))}
for g, name in [(0, "women"), (1, "men")]:
    d = per_person[per_person["gender"] == g]
    corr_self[name] = float(d["self_attr"].corr(d["looks_received"]))
corr_received = float(per_person["looks_received"].corr(per_person["yes_received"]))
pair_corr = {
    "her_yes_vs_his_self_rated_looks": float(pairs["dec_W"].corr(pairs["attr3_1_M"])),
    "her_yes_vs_her_rating_of_his_looks": float(pairs["dec_W"].corr(pairs["card_attr_W"])),
    "his_yes_vs_her_self_rated_looks": float(pairs["dec_M"].corr(pairs["attr3_1_W"])),
    "his_yes_vs_his_rating_of_her_looks": float(pairs["dec_M"].corr(pairs["card_attr_M"])),
}
side = {}
for tgt, label in [("dec_W", "her yes"), ("dec_M", "his yes")]:
    yy = pairs[tgt].to_numpy()
    m = Pipeline([("pre", preprocessor()), ("lr", sk_model(BEST_C))]).fit(X_all[is_dev], yy[is_dev])
    side[label] = {"test_auc": float(roc_auc_score(yy[is_test], m.predict_proba(X_all[is_test])[:, 1])),
                   "yes_rate": float(yy.mean())}
summary["attraction"] = {"whatif": whatif.to_dict("records"), "corr_self_rated_vs_received_looks": corr_self,
                         "corr_received_looks_vs_yes_received": corr_received, "pair_level_corr": pair_corr,
                         "one_sided_models": side, "people": per_person.dropna().reset_index().to_dict("records")}
log(f"self-rated looks vs looks received: r = {corr_self['all']:.2f} (women {corr_self['women']:.2f}, "
    f"men {corr_self['men']:.2f}); looks received vs yes received: r = {corr_received:.2f}")
log(f"one-sided models, test AUC: her yes {side['her yes']['test_auc']:.3f}, his yes {side['his yes']['test_auc']:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
ax[0].barh(whatif["information"][::-1], whatif["roc_auc"][::-1], color=["#8d8d8d", "#ababab", "#c9c9c9", "#2a6fdb"])
ax[0].axvline(0.5, color="#666", lw=0.8, ls="--")
for i, v in enumerate(whatif["roc_auc"][::-1]):
    ax[0].text(v + 0.005, i, f"{v:.2f}", va="center")
ax[0].set(xlim=(0.45, 0.9), xlabel="test ROC AUC", title="What seeing attraction would add")
pp = per_person.dropna()
jit = np.random.default_rng(0).normal(0, 0.12, len(pp))
for g, col, name in [(0, "#d1495b", "women"), (1, "#2a6fdb", "men")]:
    s = (pp["gender"] == g).to_numpy()
    ax[1].scatter(pp["self_attr"][s] + jit[s], pp["looks_received"][s], s=7, alpha=0.5, color=col,
                  label=f"{name} (r = {corr_self[name]:.2f})")
ax[1].set(xlabel="self-rated attractiveness (sign-up)", ylabel="mean attractiveness rating received",
          title="Self-ratings say little about looks")
ax[1].legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "attraction_gap.png")
plt.show()

## 9c. How much data is enough?
Train on random subsets of the 13 training events (whole events, 40 draws per size), score on the 3 validation events. The test events stay untouched.

In [ ]:
rng = np.random.default_rng(SEED)
train_waves = np.unique(wave[is_train])
lc_rows = []
for k in [2, 4, 6, 8, 10, 12, 13]:
    for _ in range(40 if k < 13 else 1):
        rows = np.isin(wave, rng.choice(train_waves, size=k, replace=False))
        m = Pipeline([("pre", preprocessor()), ("lr", sk_model(BEST_C))]).fit(X_all[rows], y_all[rows])
        lc_rows.append({"events": k, "pairs": int(rows.sum()),
                        "train_auc": roc_auc_score(y_all[rows], m.predict_proba(X_all[rows])[:, 1]),
                        "val_auc": roc_auc_score(y_val, m.predict_proba(X_all[is_val])[:, 1])})
lc = pd.DataFrame(lc_rows)
lc_sum = lc.groupby("events").agg(pairs=("pairs", "mean"), train_auc=("train_auc", "mean"),
                                  val_auc=("val_auc", "mean"), val_auc_sd=("val_auc", "std")).reset_index()
upper = lc_sum[lc_sum["events"] >= 6]
slope = float(np.polyfit(np.log2(upper["pairs"]), upper["val_auc"], 1)[0])
n_pos_dev = int(y_dev.sum())
summary["learning_curve"] = {"points": lc.to_dict("records"), "summary": lc_sum.to_dict("records"),
                             "auc_gain_per_doubling": slope, "events_per_variable_dev": n_pos_dev / len(FEATURES),
                             "positives_dev": n_pos_dev}
log(f"validation AUC gain per doubling of training pairs (6+ events): {slope:+.3f}; "
    f"matches per feature in development data: {n_pos_dev / len(FEATURES):.1f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(lc["pairs"], lc["val_auc"], s=6, alpha=0.3, color="#2a6fdb")
ax.plot(lc_sum["pairs"], lc_sum["val_auc"], marker="o", color="#2a6fdb", label="validation AUC (mean of 40 draws)")
ax.plot(lc_sum["pairs"], lc_sum["train_auc"], marker="o", color="#999", label="training AUC")
ax.axhline(0.5, color="#666", lw=0.8, ls="--")
ax.set(xscale="log", xlabel="training pairs (log scale, whole events)", ylabel="ROC AUC", title="More events help, slowly")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "learning_curve.png")
plt.show()
lc_sum.round(3)

### Takeaways
- **The three implementations agree.** Same objective, same minimum: coefficients and test probabilities match to about 1e-6, and they take the same seat/don't-seat decision on the test pairs.
- **Sign-up data carries a weak but real signal.** Test ROC AUC is about 0.56 (bootstrap 95% interval roughly 0.51-0.61). At the cost-derived threshold of 0.2 the model costs about EUR 2,745 on the test events, against EUR 2,820 for seating nobody and EUR 3,930 for seating everyone.
- **Where it pays off is the seating plan.** When everyone meets half the room, the model's plan keeps 81 of 141 matches against about 72 for a random plan.
- **The missing ingredient is pairwise attraction.** Knowing how attractive the two found each other lifts AUC to about 0.74; the sign-up form's self-rated looks barely track the ratings people actually receive.
- **More events would help, slowly** (about +0.01 AUC per doubling of training data).

## 10. Save everything the dashboard needs
The dashboard below loads only these files, so it would work the same in a fresh session (or as `app.ipynb` in the repo).

In [ ]:
joblib.dump(pre, ART / "preprocessor.joblib")
joblib.dump(sk, ART / "sklearn_logreg.joblib")
torch.save({"w": manual_params["w"], "b": manual_params["b"], "lam": lam, "lr": BEST_LR["manual"], "epochs": EPOCHS},
           ART / "torch_manual.pt")
torch.save({"state_dict": module.state_dict(), "n_features": len(FEATURES), "lam": lam, "lr": BEST_LR["module"],
            "epochs": EPOCHS}, ART / "torch_module.pt")

keep = ["iid_W", "iid_M", "wave", "dec_W", "dec_M", "match"]
pairs.loc[is_dev, keep].join(X_all[is_dev]).to_csv(ART / "dev_pairs.csv", index=False)
pairs.loc[is_test, keep].join(X_all[is_test]).to_csv(ART / "test_pairs.csv", index=False)
pd.DataFrame({"iid_W": test_pairs["iid_W"], "iid_M": test_pairs["iid_M"],
              **{f"p_{k}": v for k, v in preds.items()}}).to_csv(ART / "test_predictions.csv", index=False)
coefs.to_csv(ART / "coefficients.csv", index=False)
summary.update(features=FEATURES, sklearn_objective_final=J_final["sklearn"],
               loss_history={"manual": manual_hist[::10], "module": module_hist[::10], "every": 10})
with open(ART / "summary.json", "w") as f:
    json.dump(summary, f, indent=1, default=float)
log(f"saved: {', '.join(sorted(p.name for p in ART.iterdir()))}")

## 11. Dashboard
Everything below is loaded from `artifacts/`. Nothing is trained: the three models are read from disk and only used for inference on the held-out test events. The helper functions defined above (`evaluate`, `seating_plan`, `describe`, ...) are reused; the training variables are not.

In [ ]:
import gradio as gr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

with open(ART / "summary.json") as f:
    S = json.load(f)
APP_FEATURES: list[str] = S["features"]
PRE = joblib.load(ART / "preprocessor.joblib")
SK = joblib.load(ART / "sklearn_logreg.joblib")
MANUAL = torch.load(ART / "torch_manual.pt", weights_only=True)
_ckpt = torch.load(ART / "torch_module.pt", weights_only=True)
MODULE = LogisticRegressionNet(_ckpt["n_features"])
MODULE.load_state_dict(_ckpt["state_dict"])
MODULE.eval()

TEST = pd.read_csv(ART / "test_pairs.csv")
DEV = pd.read_csv(ART / "dev_pairs.csv")
XS_TEST = PRE.transform(TEST[APP_FEATURES])
Y_TEST = TEST["match"].to_numpy()

MODELS = {"sklearn": "scikit-learn", "manual": "PyTorch manual loop", "module": "PyTorch nn.Module + Adam"}
PRED = {"sklearn": SK.predict_proba(XS_TEST)[:, 1], "manual": predict_manual(MANUAL, XS_TEST),
        "module": predict_module(MODULE, XS_TEST)}
COEF = pd.DataFrame({"feature": APP_FEATURES, "sklearn": SK.coef_.ravel(),
                     "manual": MANUAL["w"].numpy().astype(float),
                     "module": MODULE.linear.weight.detach().numpy().ravel().astype(float)})
INTERCEPT = {"sklearn": float(SK.intercept_[0]), "manual": float(MANUAL["b"].item()),
             "module": float(MODULE.linear.bias.item())}

# The loaded models must reproduce the predictions saved at training time.
_saved = pd.read_csv(ART / "test_predictions.csv")
for k in PRED:
    gap = float(np.abs(PRED[k] - _saved[f"p_{k}"].to_numpy()).max())
    assert gap < 1e-5, f"loaded {k} model does not reproduce training-time predictions (max gap {gap:.2e})"

APP_COST_FP, APP_COST_FN = S["threshold"]["cost_fp"], S["threshold"]["cost_fn"]
APP_T = S["threshold"]["t_star"]
TEST_WAVES = sorted(int(w) for w in TEST["wave"].unique())
LAYOUT = dict(template="plotly_white", margin=dict(l=60, r=20, t=50, b=50), font=dict(size=12))


def fmt_eur(x: float) -> str:
    return f"EUR {x:,.0f}"


print(f"loaded 3 models, {len(TEST)} test pairs from waves {TEST_WAVES[0]}-{TEST_WAVES[-1]}; predictions reproduced")

### Tab 1: the three implementations

In [ ]:
def metrics_table() -> pd.DataFrame:
    base = float(S["threshold"]["dev_base_rate"])
    rows = []
    for label, p in [*((MODELS[k], PRED[k]) for k in MODELS),
                     ("baseline: everyone gets the base rate", np.full(len(Y_TEST), base)),
                     ("baseline: schedule everyone", np.ones(len(Y_TEST)))]:
        r = evaluate(Y_TEST, p, APP_T, APP_COST_FP, APP_COST_FN)
        rows.append({"model": label, "ROC AUC": r["roc_auc"], "PR AUC": r["pr_auc"], "log loss": r["log_loss"],
                     "Brier": r["brier"], f"cost at t={APP_T:.2f}": fmt_eur(r["cost_eur"])})
    return pd.DataFrame(rows).round(4)


def agreement_table() -> pd.DataFrame:
    rows = []
    keys = list(MODELS)
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            rows.append({
                "pair": f"{MODELS[a]} vs {MODELS[b]}",
                "max |coef diff|": f"{np.abs(COEF[a] - COEF[b]).max():.2e}",
                "|intercept diff|": f"{abs(INTERCEPT[a] - INTERCEPT[b]):.2e}",
                "max |prob diff| (test)": f"{np.abs(PRED[a] - PRED[b]).max():.2e}",
                "same decision at t*": f"{np.mean((PRED[a] >= APP_T) == (PRED[b] >= APP_T)):.1%}",
            })
    return pd.DataFrame(rows)


def fig_pred_vs_actual(n_bins: int = 8) -> go.Figure:
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=(
        "Predicted vs actual match rate (test events, equal-size bins)", "Predicted probability by what actually happened"))
    widths = {"sklearn": 6, "manual": 3.5, "module": 1.5}
    dashes = {"sklearn": "solid", "manual": "dash", "module": "dot"}
    for k, label in MODELS.items():
        cal = calibration_table(Y_TEST, PRED[k], int(n_bins))
        fig.add_trace(go.Scatter(
            x=cal["mean_pred"], y=cal["match_rate"], mode="lines+markers", name=label,
            line=dict(color=C3[k], width=widths[k], dash=dashes[k]),
            error_y=dict(type="data", array=1.96 * cal["se"], visible=(k == "sklearn"), color=C3[k]),
            hovertemplate="mean predicted %{x:.3f}<br>observed %{y:.3f}<br>n=%{text}<extra>" + label + "</extra>",
            text=cal["n"]), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 0.4], y=[0, 0.4], mode="lines", name="perfect calibration",
                             line=dict(color="#999", dash="dash", width=1)), row=1, col=1)
    for k, label in MODELS.items():
        for outcome, name in [(0, "no match"), (1, "match")]:
            fig.add_trace(go.Box(y=PRED[k][Y_TEST == outcome], x=[name] * int((Y_TEST == outcome).sum()),
                                 name=label, legendgroup=k, showlegend=False, marker_color=C3[k],
                                 offsetgroup=k, boxpoints=False), row=1, col=2)
    fig.update_layout(boxmode="group", height=430, legend=dict(orientation="h", y=-0.2), **LAYOUT)
    fig.update_xaxes(title_text="mean predicted probability", range=[0, 0.4], row=1, col=1)
    fig.update_yaxes(title_text="observed match rate", range=[0, 0.4], row=1, col=1)
    fig.update_yaxes(title_text="predicted P(match)", row=1, col=2)
    return fig


def fig_coefficients() -> go.Figure:
    c = COEF.assign(desc=[describe(f) for f in COEF["feature"]],
                    group=[group_of(f) for f in COEF["feature"]]).sort_values("sklearn")
    fig = go.Figure()
    fig.add_trace(go.Bar(y=c["feature"], x=c["sklearn"], orientation="h", name="scikit-learn",
                         marker_color=C3["sklearn"], customdata=c[["desc", "group"]],
                         hovertemplate="%{y}: %{x:.4f}<br>%{customdata[0]}<br>group: %{customdata[1]}<extra></extra>"))
    for k, sym in [("manual", "diamond"), ("module", "x")]:
        fig.add_trace(go.Scatter(y=c["feature"], x=c[k], mode="markers", name=MODELS[k],
                                 marker=dict(color=C3[k], symbol=sym, size=8)))
    fig.update_layout(title="Coefficients on standardised features (log-odds per 1 SD)", height=900, **LAYOUT)
    return fig


def fig_loss_curves() -> go.Figure:
    h = S["loss_history"]
    fig = go.Figure()
    for k in ("manual", "module"):
        gap = np.maximum(np.array(h[k]) - S["sklearn_objective_final"], 1e-9)
        fig.add_trace(go.Scatter(x=np.arange(len(gap)) * h["every"], y=gap, mode="lines", name=MODELS[k],
                                 line=dict(color=C3[k])))
    fig.update_layout(title="Training objective minus scikit-learn's minimum (log scale, floored at 1e-9)",
                      xaxis_title="epoch", yaxis_title="gap", yaxis_type="log", height=360, **LAYOUT)
    return fig


def lr_table() -> pd.DataFrame:
    t = pd.DataFrame(S["lr_search"])
    t["method"] = t["method"].map({"manual": "manual loop (plain gradient descent)", "module": "nn.Module + Adam"})
    return t.round(6)

### Tab 2: threshold and business cost

In [ ]:
def fig_confusion(cm: dict) -> go.Figure:
    z = [[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]]
    text = [[f"{cm['tn']}<br>not seated, no match", f"{cm['fp']}<br>seated, no match (FP)"],
            [f"{cm['fn']}<br>missed match (FN)", f"{cm['tp']}<br>seated, matched (TP)"]]
    fig = go.Figure(go.Heatmap(z=z, x=["not seated", "seated"], y=["no match", "match"], text=text,
                               texttemplate="%{text}", colorscale="Blues", showscale=False))
    fig.update_layout(title=f"Confusion matrix on the {len(Y_TEST)} test pairs", xaxis_title="decision",
                      yaxis_title="what happened", height=380, **LAYOUT)
    return fig


def threshold_view(model_key: str, t: float, c_fp: float, c_fn: float):
    c_fp, c_fn = float(c_fp or 0), float(c_fn or 0)
    p = PRED[model_key]
    cm = confusion(Y_TEST, p, t)
    cost = business_cost(cm, c_fp, c_fn)
    everyone = business_cost(confusion(Y_TEST, np.ones(len(p)), 0.5), c_fp, c_fn)
    nobody = business_cost(confusion(Y_TEST, np.zeros(len(p)), 0.5), c_fp, c_fn)
    t_opt = cost_threshold(c_fp, c_fn) if (c_fp + c_fn) > 0 else 0.5
    curve = cost_curve(Y_TEST, p, c_fp, c_fn)
    best = curve.loc[curve["cost_eur"].idxmin()]
    seated = cm["tp"] + cm["fp"]
    md = (
        f"### Cost at threshold {t:.3f}: **{fmt_eur(cost)}**\n"
        f"- schedule everyone (today's full rotation): {fmt_eur(everyone)}  \n"
        f"- schedule nobody / predict 'no match' for all: {fmt_eur(nobody)}  \n"
        f"- saving vs the better of the two: **{fmt_eur(min(everyone, nobody) - cost)}**"
        f"{' (negative: the model loses money at this threshold)' if cost > min(everyone, nobody) else ''}\n\n"
        f"Seated **{seated}** of {len(p)} pairs ({seated / len(p):.0%}); they contain **{cm['tp']}** of the "
        f"{int(Y_TEST.sum())} real matches (recall {cm['tp'] / max(1, Y_TEST.sum()):.0%}, "
        f"precision {cm['tp'] / max(1, seated):.0%}).\n\n"
        f"Decision rule for these costs: seat a pair when P(match) > c_fp / (c_fp + c_fn) = **{t_opt:.3f}**. "
        f"Lowest cost on the test curve: {fmt_eur(best['cost_eur'])} at t = {best['threshold']:.3f}."
    )
    fig = go.Figure()
    for k, label in MODELS.items():
        cc = cost_curve(Y_TEST, PRED[k], c_fp, c_fn)
        fig.add_trace(go.Scatter(x=cc["threshold"], y=cc["cost_eur"], mode="lines", name=label,
                                 line=dict(color=C3[k], width=4 if k == model_key else 1.5)))
    fig.add_hline(y=everyone, line=dict(color="#999", dash="dot"), annotation_text="schedule everyone")
    fig.add_hline(y=nobody, line=dict(color="#555", dash="dash"), annotation_text="schedule nobody")
    fig.add_vline(x=t, line=dict(color="#d1495b"), annotation_text="your threshold")
    fig.add_vline(x=t_opt, line=dict(color="#333", dash="dot"), annotation_text="c_fp/(c_fp+c_fn)",
                  annotation_position="bottom right")
    fig.update_layout(title="Business cost on test events vs decision threshold", xaxis_title="threshold",
                      yaxis_title="cost (EUR)", xaxis_range=[0, max(0.5, t + 0.05, t_opt + 0.05)], height=420, **LAYOUT)
    return fig_confusion(cm), md, fig


def threshold_for_costs(c_fp: float, c_fn: float) -> float:
    """When the costs change, move the slider to the cost-optimal threshold."""
    c_fp, c_fn = float(c_fp or 0), float(c_fn or 0)
    return round(cost_threshold(c_fp, c_fn), 3) if (c_fp + c_fn) > 0 else 0.5

### Tab 3: feature and target distributions

In [ ]:
def feature_view(col: str):
    d = DEV[[col, "match"]].dropna()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for outcome, name, color in [(0, "no match", "#9aa5b1"), (1, "match", "#d1495b")]:
        fig.add_trace(go.Histogram(x=d.loc[d["match"] == outcome, col], name=name, marker_color=color, opacity=0.65,
                                   histnorm="probability density", nbinsx=30), secondary_y=False)
    if d[col].nunique() > 6:   # match rate by quantile bin of the feature
        rate = d.groupby(pd.qcut(d[col], 8, duplicates="drop"), observed=True).agg(
            x=(col, "mean"), rate=("match", "mean"), n=("match", "size"))
    else:
        rate = d.groupby(col).agg(x=(col, "mean"), rate=("match", "mean"), n=("match", "size"))
    fig.add_trace(go.Scatter(x=rate["x"], y=rate["rate"], mode="lines+markers", name="match rate",
                             line=dict(color="#222"), text=rate["n"],
                             hovertemplate="%{x:.2f}: match rate %{y:.3f} (n=%{text})<extra></extra>"), secondary_y=True)
    fig.update_layout(barmode="overlay", title=f"{col}: {describe(col)} (development events, before imputation)",
                      height=420, legend=dict(orientation="h", y=-0.2), **LAYOUT)
    fig.update_yaxes(title_text="density", secondary_y=False)
    fig.update_yaxes(title_text="match rate", secondary_y=True, rangemode="tozero", tickformat=".2f")
    coef = COEF.set_index("feature").loc[col, "sklearn"]
    md = (f"**{col}** - {describe(col)}. Group: *{group_of(col)}*. Missing before imputation: "
          f"{DEV[col].isna().mean():.1%}. Model coefficient (per 1 SD): **{coef:+.4f}** (odds x{np.exp(coef):.3f}).")
    return fig, md


def fig_target() -> go.Figure:
    allp = pd.concat([DEV, TEST])
    g = allp.groupby("wave").agg(rate=("match", "mean"), her=("dec_W", "mean"), his=("dec_M", "mean"),
                                 n=("match", "size"), size=("event_size", "first")).reset_index()
    split = np.select([g["wave"].isin(S["split"]["test"]["waves"]), g["wave"].isin(S["split"]["validation"]["waves"])],
                      ["test", "validation"], "train")
    fig = go.Figure()
    for name, color in [("train", "#2a6fdb"), ("validation", "#e8833a"), ("test", "#2ca58d")]:
        m = split == name
        fig.add_trace(go.Bar(x=g["wave"][m], y=g["rate"][m], name=f"mutual match rate ({name})", marker_color=color,
                             customdata=np.c_[g["n"][m], g["size"][m]],
                             hovertemplate="wave %{x}: %{y:.3f}<br>%{customdata[0]} pairs, "
                                           "%{customdata[1]} partners each<extra></extra>"))
    fig.add_trace(go.Scatter(x=g["wave"], y=g["her"], mode="lines+markers", name="her yes rate", line=dict(color="#d1495b")))
    fig.add_trace(go.Scatter(x=g["wave"], y=g["his"], mode="lines+markers", name="his yes rate", line=dict(color="#555")))
    fig.update_layout(title=f"Target by event: mutual matches are rare ({S['data']['match_rate']:.1%}) and vary between nights",
                      xaxis_title="wave (event, in date order)", yaxis_title="rate", height=420,
                      legend=dict(orientation="h", y=-0.25), **LAYOUT)
    return fig

### Tab 4: event planner

In [ ]:
def planner_view(wave: int, share: float, model_key: str):
    in_ev = (TEST["wave"] == int(wave)).to_numpy()
    ev = TEST[in_ev].reset_index(drop=True)
    p = PRED[model_key][in_ev]
    y = ev["match"].to_numpy()
    chosen = seating_plan(ev, p, share)
    rng = np.random.default_rng(0)
    rand = np.array([y[seating_plan(ev, rng.random(len(ev)), share)].sum() for _ in range(60)])
    perfect = y[seating_plan(ev, y + rng.random(len(ev)) * 1e-3, share)].sum()

    women, men = sorted(ev["iid_W"].unique()), sorted(ev["iid_M"].unique())
    wi, mi = {w: i for i, w in enumerate(women)}, {m: j for j, m in enumerate(men)}
    grid = np.full((len(women), len(men)), np.nan)
    for r, prob in zip(ev.itertuples(), p):
        grid[wi[r.iid_W], mi[r.iid_M]] = prob
    fig = go.Figure(go.Heatmap(z=grid, x=[f"M{m}" for m in men], y=[f"W{w}" for w in women], colorscale="Blues",
                               colorbar=dict(title="P(match)"), hovertemplate="%{y} x %{x}: %{z:.3f}<extra></extra>"))
    seat, hit = ev[chosen], ev[ev["match"] == 1]
    fig.add_trace(go.Scatter(x=[f"M{m}" for m in seat["iid_M"]], y=[f"W{w}" for w in seat["iid_W"]], mode="markers",
                             name="seated by the model",
                             marker=dict(symbol="square-open", size=13, color="#e8833a", line=dict(width=2))))
    fig.add_trace(go.Scatter(x=[f"M{m}" for m in hit["iid_M"]], y=[f"W{w}" for w in hit["iid_W"]], mode="markers",
                             name="real mutual match", marker=dict(symbol="circle", size=7, color="#d1495b")))
    fig.update_layout(title=f"Wave {wave}: predicted P(match) for every woman x man pair",
                      height=max(420, 22 * len(women) + 160), legend=dict(orientation="h", y=-0.12), **LAYOUT)
    md = (f"### Wave {wave}: {len(women)} women x {len(men)} men, {int(y.sum())} real matches\n"
          f"Each woman meets {int(np.ceil(share * len(men)))} of {len(men)} men "
          f"({int(chosen.sum())} dates instead of {len(ev)}).\n\n"
          f"| plan | matches kept |\n|---|---|\n"
          f"| model ({MODELS[model_key]}) | **{int(y[chosen].sum())}** |\n"
          f"| random seating (mean of 60 plans) | {rand.mean():.1f} (range {rand.min()}-{rand.max()}) |\n"
          f"| perfect foresight | {int(perfect)} |\n\n"
          f"Share of random plans doing at least as well as the model: {np.mean(rand >= y[chosen].sum()):.0%}.")
    return fig, md


def planner_summary_md() -> str:
    rows = ["| share of the room each person meets | dates held | model plan | random plan (mean +/- sd) "
            "| perfect plan | random plans at least as good |", "|---|---|---|---|---|---|"]
    for share, r in S["planner"]["by_share"].items():
        rows.append(f"| {float(share):.0%} | {r['dates_held']} | **{r['model_plan']}** | "
                    f"{r['random_plan']:.1f} +/- {r['random_plan_sd']:.1f} | {r['perfect_plan']} | "
                    f"{r['share_random_draws_at_least_as_good']:.0%} |")
    n_matches = next(iter(S["planner"]["by_share"].values()))["all_matches"]
    return f"All {len(TEST_WAVES)} test events together ({n_matches} real matches), scikit-learn model:\n\n" + "\n".join(rows)

### Tab 5: what physical attraction would add

In [ ]:
def fig_whatif() -> go.Figure:
    w = pd.DataFrame(S["attraction"]["whatif"])
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Test ROC AUC", "Matches kept, half-room seating plan"))
    colors = ["#2a6fdb", "#c9c9c9", "#ababab", "#8d8d8d"]
    fig.add_trace(go.Bar(y=w["information"], x=w["roc_auc"], orientation="h", marker_color=colors,
                         text=w["roc_auc"].round(3), textposition="outside", showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(y=w["information"], x=w["planner_matches"], orientation="h", marker_color=colors,
                         text=w["planner_matches"], textposition="outside", showlegend=False), row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=2)
    fig.update_xaxes(range=[0.45, 0.9], row=1, col=1)
    fig.update_layout(height=380, **LAYOUT)
    return fig


def fig_self_vs_received() -> go.Figure:
    pp = pd.DataFrame(S["attraction"]["people"])
    r = S["attraction"]["corr_self_rated_vs_received_looks"]
    rng = np.random.default_rng(0)
    fig = go.Figure()
    for g, name, color in [(0, "women", "#d1495b"), (1, "men", "#2a6fdb")]:
        d = pp[pp["gender"] == g]
        fig.add_trace(go.Scatter(x=d["self_attr"] + rng.normal(0, 0.12, len(d)), y=d["looks_received"], mode="markers",
                                 name=f"{name} (r = {r[name]:.2f})", marker=dict(color=color, size=6, opacity=0.55)))
    fig.update_layout(title="Self-rated attractiveness (sign-up) vs average rating received on the night",
                      xaxis_title="self-rated attractiveness (jittered)", yaxis_title="mean rating received",
                      height=420, **LAYOUT)
    return fig


def attraction_md() -> str:
    a = S["attraction"]
    pc, side, cs = a["pair_level_corr"], a["one_sided_models"], a["corr_self_rated_vs_received_looks"]
    return (
        "**The one looks signal we have before the event is a self-rating, and it barely tracks what others see** "
        f"(r = {cs['all']:.2f} overall, {cs['women']:.2f} for women, {cs['men']:.2f} for men). Meanwhile the "
        f"attractiveness rating a person receives correlates r = {a['corr_received_looks_vs_yes_received']:.2f} "
        "with the share of partners who say yes to them.\n\n"
        "| pair-level correlation with a yes | her yes | his yes |\n|---|---|---|\n"
        f"| partner's self-rated looks (sign-up) | {pc['her_yes_vs_his_self_rated_looks']:.3f} "
        f"| {pc['his_yes_vs_her_self_rated_looks']:.3f} |\n"
        f"| how attractive they found the partner (scorecard) | {pc['her_yes_vs_her_rating_of_his_looks']:.3f} "
        f"| {pc['his_yes_vs_his_rating_of_her_looks']:.3f} |\n\n"
        "Same features and model, predicting one side's yes instead of the mutual match: test AUC "
        f"**{side['her yes']['test_auc']:.3f}** for her decision, **{side['his yes']['test_auc']:.3f}** for his.\n\n"
        "Variants 2-4 use information from the night itself, so they are *not* deployable; they show the size of the "
        "gap. The looks reputation says who is popular, but a fair seating plan gives everyone the same number of "
        "dates anyway, so for the planner only *pairwise* attraction (variant 3 onwards) matters."
    )

### Tab 6: data size and split

In [ ]:
def fig_learning_curve() -> go.Figure:
    lc = pd.DataFrame(S["learning_curve"]["points"])
    sm = pd.DataFrame(S["learning_curve"]["summary"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lc["pairs"], y=lc["val_auc"], mode="markers", name="one random draw of events",
                             marker=dict(color="#2a6fdb", size=5, opacity=0.3)))
    fig.add_trace(go.Scatter(x=sm["pairs"], y=sm["val_auc"], mode="lines+markers", name="validation AUC (mean)",
                             line=dict(color="#2a6fdb", width=3)))
    fig.add_trace(go.Scatter(x=sm["pairs"], y=sm["train_auc"], mode="lines+markers", name="training AUC (mean)",
                             line=dict(color="#999", width=2)))
    fig.add_hline(y=0.5, line=dict(color="#666", dash="dash"))
    fig.update_layout(title="Learning curve: training on 2 to 13 whole events, scored on the 3 validation events",
                      xaxis_title="training pairs (log scale)", xaxis_type="log", yaxis_title="ROC AUC",
                      height=420, **LAYOUT)
    return fig


def data_size_md() -> str:
    d, lc, lk = S["data"], S["learning_curve"], S["leakage_check"]
    lo, hi = S["bootstrap"]["auc_ci"]["sklearn"]
    return (
        f"- **{d['date_records']:,}** date records = **{d['pairs']:,}** unique pairs, but only **{d['people']}** people "
        f"and **{d['events']}** events. Each person appears in ~{2 * d['pairs'] / d['people']:.0f} pairs, so rows are far "
        "from independent.\n"
        f"- **{d['matches']}** positives in total; {lc['positives_dev']} in the development events, i.e. "
        f"**{lc['events_per_variable_dev']:.1f} matches per feature** (rule of thumb for logistic regression: 10-20).\n"
        f"- Test set: {S['split']['test']['pairs']} pairs with {S['split']['test']['matches']} matches. Bootstrap 95% "
        f"interval for the test AUC: **{lo:.3f} - {hi:.3f}**. Differences between feature sets smaller than ~0.05 AUC "
        "cannot be told apart on this test set.\n"
        f"- Learning curve: about **{lc['auc_gain_per_doubling']:+.3f} AUC per doubling** of training pairs from 6 "
        "events up. The training-validation gap is still closing, so more events would help, slowly.\n"
        f"- Random-row split vs whole-event split (5-fold CV, same model): AUC **{lk['random_rows_auc_mean']:.3f}** vs "
        f"**{lk['by_event_auc_mean']:.3f}**. With random rows {lk['share_eval_pairs_with_person_seen_in_training']:.0%} "
        "of evaluation pairs involve someone the model already saw in training."
    )

### Tab 7: pair explorer

In [ ]:
def women_in(wave: int) -> list:
    return sorted(int(w) for w in TEST.loc[TEST["wave"] == int(wave), "iid_W"].unique())


def men_for(wave: int, woman: int) -> list:
    m = (TEST["wave"] == int(wave)) & (TEST["iid_W"] == int(woman))
    return sorted(int(x) for x in TEST.loc[m, "iid_M"].unique())


def on_wave(wave):
    ws = women_in(wave)
    ms = men_for(wave, ws[0])
    return gr.Dropdown(choices=ws, value=ws[0]), gr.Dropdown(choices=ms, value=ms[0])


def on_woman(wave, woman):
    ms = men_for(wave, woman)
    return gr.Dropdown(choices=ms, value=ms[0])


def pair_view(wave, woman, man):
    m = ((TEST["wave"] == int(wave)) & (TEST["iid_W"] == int(woman)) & (TEST["iid_M"] == int(man))).to_numpy()
    if not m.any():
        return go.Figure(), "Pick a pair."
    i = int(np.flatnonzero(m)[0])
    row = TEST.iloc[i]
    contrib = XS_TEST[i] * COEF["sklearn"].to_numpy()
    c = pd.DataFrame({"feature": APP_FEATURES, "value": row[APP_FEATURES].to_numpy(dtype=float), "contribution": contrib})
    c = c.reindex(c["contribution"].abs().sort_values(ascending=False).index).head(12).iloc[::-1]
    fig = go.Figure(go.Bar(y=c["feature"], x=c["contribution"], orientation="h",
                           marker_color=np.where(c["contribution"] > 0, "#2ca58d", "#d1495b"),
                           customdata=np.c_[c["value"], [describe(f) for f in c["feature"]]],
                           hovertemplate="%{y} = %{customdata[0]:.2f}<br>%{customdata[1]}"
                                         "<br>log-odds %{x:+.3f}<extra></extra>"))
    fig.update_layout(title="Top 12 contributions to the log-odds (scikit-learn): coefficient x standardised value",
                      xaxis_title="push towards match (+) or no match (-)", height=440, **LAYOUT)
    yes = lambda v: "yes" if v == 1 else "no"
    md = (f"### Woman {woman} x man {man} (wave {wave})\n| | P(match) |\n|---|---|\n"
          + "".join(f"| {MODELS[k]} | **{PRED[k][i]:.4f}** |\n" for k in MODELS)
          + f"| base rate (development events) | {S['threshold']['dev_base_rate']:.4f} |\n\n"
          f"What happened: she said **{yes(row['dec_W'])}**, he said **{yes(row['dec_M'])}** -> "
          f"{'**mutual match**' if row['match'] == 1 else 'no match'}.\n\n"
          f"Intercept {INTERCEPT['sklearn']:+.3f}; sum of all contributions {contrib.sum():+.3f}. "
          "Missing answers were filled with the training median before scaling.")
    return fig, md

### Build and launch the app

In [ ]:
n_dev_events = len(S["split"]["train"]["waves"]) + len(S["split"]["validation"]["waves"])
INTRO = f"""
# Who will say yes to each other? A pre-event match model for speed-dating nights

An organiser gets everyone's sign-up survey before the night. If the night is too short for full rotation,
which pairs should meet? This dashboard serves a logistic regression trained **three ways** (scikit-learn,
a hand-written PyTorch loop, and the standard `nn.Module` + `torch.optim` workflow) on {n_dev_events} past events and
evaluated on {len(TEST_WAVES)} later ones it never saw. All models are loaded from `artifacts/`; nothing is retrained here.

Business defaults: a seated pair that does not match costs **EUR {APP_COST_FP:.0f}** (a wasted 4-minute slot for two
people), a match we fail to seat costs **EUR {APP_COST_FN:.0f}**. That puts the decision threshold at
**{APP_T:.2f}**. The costs are editable in tab 2.
"""


def build_demo() -> gr.Blocks:
    waves = TEST_WAVES
    w0 = women_in(waves[-1])[0]
    m0 = men_for(waves[-1], w0)[0]
    with gr.Blocks(title="Speed-dating match model") as demo:
        gr.Markdown(INTRO)
        with gr.Tab("1. Three implementations"):
            gr.Markdown(f"### Test events (waves {waves[0]}-{waves[-1]}): same split, same features, same objective")
            gr.Dataframe(value=metrics_table(), interactive=False)
            gr.Markdown("### Do they agree?")
            gr.Dataframe(value=agreement_table(), interactive=False)
            bins = gr.Slider(minimum=4, maximum=15, value=8, step=1, label="number of bins for the calibration plot")
            pva = gr.Plot(value=fig_pred_vs_actual(8))
            bins.change(fig_pred_vs_actual, inputs=bins, outputs=pva)
            gr.Plot(value=fig_coefficients())
            gr.Plot(value=fig_loss_curves())
            gr.Markdown(f"### Learning-rate search (fit on the training events, {S['epochs']} full-batch epochs)\n"
                        "Chosen = lowest final training objective without diverging. "
                        f"Chosen: manual **{S['best_lr']['manual']}**, nn.Module **{S['best_lr']['module']}**; "
                        f"C = **{S['best_C']}** for all three.")
            gr.Dataframe(value=lr_table(), interactive=False)

        with gr.Tab("2. Threshold and cost"):
            with gr.Row():
                model_dd = gr.Dropdown(choices=[(v, k) for k, v in MODELS.items()], value="sklearn", label="model")
                thr = gr.Slider(minimum=0.0, maximum=1.0, value=round(APP_T, 3), step=0.005,
                                label="decision threshold: seat the pair if P(match) >= threshold")
            with gr.Row():
                cfp = gr.Number(value=APP_COST_FP, label="cost of a seated pair that does not match (EUR)", minimum=0)
                cfn = gr.Number(value=APP_COST_FN, label="cost of a match we never seat (EUR)", minimum=0)
            cm0, md0, cc0 = threshold_view("sklearn", round(APP_T, 3), APP_COST_FP, APP_COST_FN)
            with gr.Row():
                cm_plot = gr.Plot(value=cm0)
                cost_md = gr.Markdown(value=md0)
            curve_plot = gr.Plot(value=cc0)
            outs = [cm_plot, cost_md, curve_plot]
            for comp in (model_dd, thr):
                comp.change(threshold_view, inputs=[model_dd, thr, cfp, cfn], outputs=outs)
            for comp in (cfp, cfn):
                comp.change(threshold_for_costs, inputs=[cfp, cfn], outputs=thr).then(
                    threshold_view, inputs=[model_dd, thr, cfp, cfn], outputs=outs)

        with gr.Tab("3. Feature and target distributions"):
            gr.Plot(value=fig_target())
            feat = gr.Dropdown(choices=[(f"{c} - {describe(c)}", c) for c in APP_FEATURES], value="pref_fit_W",
                               label="feature")
            f0, fmd0 = feature_view("pref_fit_W")
            feat_md = gr.Markdown(value=fmd0)
            feat_plot = gr.Plot(value=f0)
            feat.change(feature_view, inputs=feat, outputs=[feat_plot, feat_md])

        with gr.Tab("4. Event planner"):
            gr.Markdown("The decision the model is for: the night is too short for everyone to meet everyone. "
                        "The model's plan picks the dates with the highest total P(match), with the same "
                        "number of dates per person.\n\n" + planner_summary_md())
            with gr.Row():
                ev_dd = gr.Dropdown(choices=waves, value=waves[-1], label="test event (wave)")
                share = gr.Slider(minimum=0.2, maximum=1.0, value=0.5, step=0.05,
                                  label="share of the other side each person meets")
                pm_dd = gr.Dropdown(choices=[(v, k) for k, v in MODELS.items()], value="sklearn", label="model")
            pf0, pmd0 = planner_view(waves[-1], 0.5, "sklearn")
            plan_md = gr.Markdown(value=pmd0)
            plan_plot = gr.Plot(value=pf0)
            for comp in (ev_dd, share, pm_dd):
                comp.change(planner_view, inputs=[ev_dd, share, pm_dd], outputs=[plan_plot, plan_md])

        with gr.Tab("5. The missing piece: attraction"):
            gr.Markdown(attraction_md())
            gr.Plot(value=fig_whatif())
            gr.Plot(value=fig_self_vs_received())

        with gr.Tab("6. Data size and split"):
            gr.Markdown(data_size_md())
            gr.Plot(value=fig_learning_curve())
            gr.Dataframe(value=pd.DataFrame(S["c_search"]).round(4), interactive=False,
                         label="C search, leave-one-event-out on the development events")
            gr.Dataframe(value=pd.DataFrame(S["ablation"]).round(4), interactive=False,
                         label="Feature-group ablation (same scoring)")
            gr.Dataframe(value=pd.DataFrame(S["tried_and_dropped"]).round(4), interactive=False,
                         label="Feature blocks tried on top of the final set and dropped (same scoring)")

        with gr.Tab("7. Pair explorer"):
            with gr.Row():
                px_wave = gr.Dropdown(choices=waves, value=waves[-1], label="test event (wave)")
                px_w = gr.Dropdown(choices=women_in(waves[-1]), value=w0, label="woman (iid)")
                px_m = gr.Dropdown(choices=men_for(waves[-1], w0), value=m0, label="man (iid)")
            xf0, xmd0 = pair_view(waves[-1], w0, m0)
            px_md = gr.Markdown(value=xmd0)
            px_plot = gr.Plot(value=xf0)
            px_wave.change(on_wave, inputs=px_wave, outputs=[px_w, px_m])
            px_w.change(on_woman, inputs=[px_wave, px_w], outputs=px_m)
            px_m.change(pair_view, inputs=[px_wave, px_w, px_m], outputs=[px_plot, px_md])
    return demo


demo = build_demo()
demo.launch(share=True)   # in Colab: inline view plus a temporary public link